In [ ]:
import pandas as pd
import numpy as np
import urllib.request
columns = [
    'duration', 'protocol_type', 'service', 'flag', 'src_bytes', 'dst_bytes',
    'land', 'wrong_fragment', 'urgent', 'hot', 'num_failed_logins', 'logged_in',
    'num_compromised', 'root_shell', 'su_attempted', 'num_root', 'num_file_creations',
    'num_shells', 'num_access_files', 'num_outbound_cmds', 'is_host_login',
    'is_guest_login', 'count', 'srv_count', 'serror_rate', 'srv_serror_rate',
    'rerror_rate', 'srv_rerror_rate', 'same_srv_rate', 'diff_srv_rate',
    'srv_diff_host_rate', 'dst_host_count', 'dst_host_srv_count',
    'dst_host_same_srv_rate', 'dst_host_diff_srv_rate', 'dst_host_same_src_port_rate',
    'dst_host_srv_diff_host_rate', 'dst_host_serror_rate', 'dst_host_srv_serror_rate',
    'dst_host_rerror_rate', 'dst_host_srv_rerror_rate', 'label', 'difficulty'
]
url = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain%2B.txt"
file_path = "network_traffic_real.csv"
print("getting NSL_KDD")
urllib.request.urlretrieve(url,file_path)
df=pd.read_csv(file_path,names=columns)
df.drop('difficulty',axis=1,inplace=True)
df['label']=df['label'].apply(lambda x: 0 if x=='normal' else 1)
print(f"Dataset successfully loaded! Shape: {df.shape}")
df.head(3)

getting NSL_KDD
Dataset successfully loaded! Shape: (125973, 42)


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,25,0.17,0.03,0.17,0.0,0.0,0.0,0.05,0.0,0
1,0,udp,other,SF,146,0,0,0,0,0,...,1,0.00,0.60,0.88,0.0,0.0,0.0,0.00,0.0,0
2,0,tcp,private,S0,0,0,0,0,0,0,...,26,0.10,0.05,0.00,0.0,1.0,1.0,0.00,0.0,1


In [ ]:
# CODE CELL 2
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import joblib

# 1. Separate Features (X) and Target (y)
X = df.drop('label', axis=1)
y = df['label']

# 2. One-Hot Encoding for categorical features
categorical_cols = ['protocol_type', 'service', 'flag']
numerical_cols = [c for c in X.columns if c not in categorical_cols]
X = pd.get_dummies(X, columns=categorical_cols, drop_first=True)

# 3. Stratified Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 4. Fit StandardScaler strictly on X_train, then transform both splits
scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

# 5. Save preprocessing artifacts in Colab environment storage
joblib.dump(scaler, "scaler.joblib")
joblib.dump(X.columns.tolist(), "feature_columns.joblib")

print("Preprocessing complete!")
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

Preprocessing complete!
X_train shape: (100778, 119) | X_test shape: (25195, 119)


In [ ]:
# CODE CELL 3
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Train Decision Tree Baseline
print("Training Decision Tree Baseline...")
dt_model = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_model.fit(X_train, y_train)

# Train Random Forest Ensemble
print("Training Random Forest Ensemble...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Save the best performing ensemble model
joblib.dump(rf_model, "best_ids_model.joblib")
print("Models trained and serialized successfully!")

Training Decision Tree Baseline...
Training Random Forest Ensemble...
Models trained and serialized successfully!


In [ ]:
# CODE CELL 4
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

dt_preds = dt_model.predict(X_test)
rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

print("=== DECISION TREE PERFORMANCE ===")
print(classification_report(y_test, dt_preds, target_names=['Normal (0)', 'Attack (1)']))

print("\n=== RANDOM FOREST PERFORMANCE ===")
print(classification_report(y_test, rf_preds, target_names=['Normal (0)', 'Attack (1)']))

print("Random Forest Confusion Matrix:")
print(confusion_matrix(y_test, rf_preds))
print(f"ROC-AUC Score: {roc_auc_score(y_test, rf_probs):.4f}")

=== DECISION TREE PERFORMANCE ===
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00     13469
  Attack (1)       1.00      1.00      1.00     11726

    accuracy                           1.00     25195
   macro avg       1.00      1.00      1.00     25195
weighted avg       1.00      1.00      1.00     25195


=== RANDOM FOREST PERFORMANCE ===
              precision    recall  f1-score   support

  Normal (0)       1.00      1.00      1.00     13469
  Attack (1)       1.00      1.00      1.00     11726

    accuracy                           1.00     25195
   macro avg       1.00      1.00      1.00     25195
weighted avg       1.00      1.00      1.00     25195

Random Forest Confusion Matrix:
[[13462     7]
 [   19 11707]]
ROC-AUC Score: 1.0000


In [ ]:
# CODE CELL 5
!pip install pyngrok flask -q

import os
from flask import Flask, request, jsonify
from pyngrok import ngrok
import threading

app = Flask(__name__)

model = joblib.load("best_ids_model.joblib")
scaler = joblib.load("scaler.joblib")
feature_cols = joblib.load("feature_columns.joblib")

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()

    # Format incoming JSON into dataframe with correct one-hot feature structure
    input_df = pd.DataFrame([data])
    input_df = input_df.reindex(columns=feature_cols, fill_value=0)

    # Scale numerical values
    numerical_cols = scaler.feature_names_in_
    input_df[numerical_cols] = scaler.transform(input_df[numerical_cols])

    prediction = model.predict(input_df)[0]
    result = "MALICIOUS (ATTACK DETECTED)" if prediction == 1 else "BENIGN (NORMAL TRAFFIC)"

    return jsonify({"status": "success", "prediction": result})

# Run Flask server in a separate background thread inside Colab
threading.Thread(target=app.run, kwargs={"port": 5002}).start()
ngrok.set_auth_token("3ISnn6iEPG2XdZ3EAspjrleqBIH_7Jt2omcNA1mSQxSs7LVoC")
# Open an HTTPS public URL tunnel using ngrok
# Sign up at ngrok.com for a free token if prompted
public_url = ngrok.connect(5002)
print(f"\n--- LIVE INFERENCE REST API API CREATED ---")
print(f"Public Endpoint URL: {public_url}/predict")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5002
INFO:werkzeug:Press CTRL+C to quit



--- LIVE INFERENCE REST API API CREATED ---
Public Endpoint URL: NgrokTunnel: "https://handheld-borrowing-browsing.ngrok-free.dev" -> "http://localhost:5002"/predict


In [ ]:
import requests

# 1. Matched to your actual active .dev ngrok domain
url = "https://handheld-borrowing-browsing.ngrok-free.dev/predict"

# 2. Sample packet data payload
sample_packet = {
    "duration": 0,
    "src_bytes": 181,
    "dst_bytes": 5450,
    "count": 8,
    "srv_count": 8,
    "serror_rate": 0.0,
    "srv_serror_rate": 0.0,
    "same_srv_rate": 1.0,
    "protocol_type_tcp": 1,
    "service_http": 1,
    "flag_SF": 1
}

# 3. Send HTTP POST request
response = requests.post(url, json=sample_packet)

print("Response Status Code:", response.status_code)
if response.status_code == 200:
    print("ML Model Output:", response.json())
else:
    print("Server returned error:", response.text)

INFO:werkzeug:127.0.0.1 - - [26/Aug/2026 18:04:59] "POST /predict HTTP/1.1" 200 -


Response Status Code: 200
ML Model Output: {'prediction': 'BENIGN (NORMAL TRAFFIC)', 'status': 'success'}


In [ ]:
!pip install scapy -q
from scapy.all import sniff, IP, TCP, UDP
import requests

def capture_and_predict(packet):
    if IP in packet:
        # Extract features dynamically from the packet headers
        dynamic_packet = {
            "duration": 0,
            "src_bytes": len(packet[IP].payload),
            "dst_bytes": len(packet),
            "count": 1,
            "srv_count": 1,
            "serror_rate": 0.0,
            "srv_serror_rate": 0.0,
            "same_srv_rate": 1.0,
            "protocol_type_tcp": 1 if TCP in packet else 0,
            "service_http": 1 if packet.haslayer(TCP) and packet[TCP].dport == 80 else 0,
            "flag_SF": 1
        }

        # Post directly to the API
        url = "https://handheld-borrowing-browsing.ngrok-free.dev/predict"
        res = requests.post(url, json=dynamic_packet)
        print("Live Packet Output:", res.json())

# Sniff 1 live network packet automatically
sniff(count=1, prn=capture_and_predict)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 26.2 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/scapy/layers/tls/crypto/groups.py:25: CryptographyDeprecationWarning: Diffie-Hellman over finite fields (FFDH) is deprecated and support will be removed in a future release. Use a more modern key exchange algorithm.
  from cryptography.hazmat.primitives.asymmetric.dh import DHParameterNumbers
INFO:werkzeug:127.0.0.1 - - [26/Aug/2026 18:06:19] "POST /predict HTTP/1.1" 200 -


Live Packet Output: {'prediction': 'BENIGN (NORMAL TRAFFIC)', 'status': 'success'}


<Sniffed: TCP:1 UDP:0 ICMP:0 Other:0>